In [2]:
import sqlite3
import pandas as pd
import anthropic

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
def run_sql_query(query):
    try:
        conn = sqlite3.connect('olist.db')
        result = pd.read_sql_query(query, conn)
        conn.close()
        return result.to_string()
    except Exception as e:
        return f"SQL Error: {str(e)}"

In [10]:
def run_agent(question):
    messages = [{"role": "user", "content": question}]

    system_prompt = """You are a data analyst for Olist, a Brazilian e-commerce platform. 
    You have access to 7 tables: orders, items, payments, reviews, customers, 
    products, and sellers. The data covers 100k orders from 2016-2018. The products table has a column called product_category_name (not product_category).
    Key columns:
    - orders: order_id, customer_id, order_status, order_purchase_timestamp, order_delivered_customer_date, order_estimated_delivery_date
    - items: order_id, product_id, seller_id, price, freight_value
    - products: product_id, product_category_name
    - reviews: order_id, review_score, review_comment_message
    - customers: customer_id, customer_unique_id, customer_state.
    When answering business questions, use the run_sql_query tool to query 
    the database. Always base your answers on the actual data, not assumptions.
    Respond clearly with the key insight first, then supporting numbers.
    The database is SQLite. Use julianday() for date arithmetic, not EXTRACT().To find repeat customers, always join orders with customers on customer_id 
    and use customer_unique_id, not customer_id, to identify unique customers."""

    tools = [
        {
            "name": "run_sql_query", #function to call when want to run
            "description": "Runs a SQL query against the Olist e-commerce database and returns the results. Use this to answer questions about orders, customers, products, delivery times, and review scores.",
            "input_schema": { #describe what the tool accepts
                "type": "object", #dictionary input
                "properties": { #what the dictionary can contain
                    "query": {
                        "type": "string" #query must be string
                        "description": "The SQL query to run"
                    }
                },
                "required": ["query"] #claude must provide this when running tool
            }
        }
    ]
    
    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            system=system_prompt,
            tools=tools,
            messages=messages
        )
        
        # if Claude is done, return the answer
        if response.stop_reason == "end_turn":
            return response.content[0].text, messages
        
        # if Claude wants to use a tool, run it
        if response.stop_reason == "tool_use":
            tool_use = next(block for block in response.content 
                            if block.type == "tool_use")
            query = tool_use.input["query"]
            result = run_sql_query(query)
            
            messages.append({"role": "assistant", "content": response.content})
            messages.append({
                "role": "user",
                "content": [{
                    "type": "tool_result",
                    "tool_use_id": tool_use.id,
                    "content": result
                }]
            })


In [11]:
print("Olist Insight Agent: ask me anything about the data")
print("Type 'quit' to exit\n")

history = []
while True:
    question = input("You: ")
    if question.lower() == 'quit':
        break
    elif question.lower() == 'show history':
        print(f"\nAgent: {history}\n")
    else:
        answer, history = run_agent(question)
        print(f"\nAgent: {answer}\n")

Olist Insight Agent: ask me anything about the data
Type 'quit' to exit



You:  whats the highest rated category



Agent: **The highest rated category is "cds_dvds_musicais" (CDs, DVDs & Music) with an average review score of 4.64 out of 5.**

This category has 14 reviews in the dataset, giving it the top rating among all product categories on the Olist platform.



You:  show history



Agent: [{'role': 'user', 'content': 'whats the highest rated category'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text="I'll query the database to find the highest rated product category based on average review scores.", type='text'), ToolUseBlock(id='toolu_01VTDMpV5vbSBdnThXMzvVvk', caller=DirectCaller(type='direct'), input={'query': '\nSELECT \n    p.product_category_name,\n    AVG(r.review_score) as avg_review_score,\n    COUNT(r.review_score) as review_count\nFROM products p\nJOIN items i ON p.product_id = i.product_id\nJOIN reviews r ON i.order_id = r.order_id\nGROUP BY p.product_category_name\nORDER BY avg_review_score DESC\nLIMIT 1\n'}, name='run_sql_query', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01VTDMpV5vbSBdnThXMzvVvk', 'content': '  product_category_name  avg_review_score  review_count\n0     cds_dvds_musicais          4.642857            14'}]}]



You:  quit
